# VERA v2 — M3 detector-box paper campaign

This notebook pulls the fresh M2 detector arrays from Drive, copies labels into writable storage, verifies the BioViL-T feature contract, and runs the nine retained M3 rows. The main row is always run first.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, json
REPO_URL = 'https://github.com/hiennguyendang/phase_2_3_4_5.git'
REPO_DIR = Path('/kaggle/working/vera_repo')
if not REPO_DIR.exists(): subprocess.run(['git','clone',REPO_URL,str(REPO_DIR)], check=True)
print('commit:', subprocess.check_output(['git','-C',str(REPO_DIR),'rev-parse','HEAD'],text=True).strip())
sys.path.insert(0, str(REPO_DIR/'kaggle_notebooks'))
from vera_common import find_bundle, find_m2_outputs, FEATURE_ROOT, configure_drive, copy_tree
bundle = find_bundle()
m2_output = find_m2_outputs()
remote = configure_drive()
print('bundle:', bundle, 'detector output:', m2_output, 'features:', FEATURE_ROOT)

In [ ]:
# Build a writable detector-label directory; never mutate /kaggle/input.
labels = Path('/kaggle/working/m3_labels_detector_v2')
if labels.exists(): shutil.rmtree(labels)
copy_tree(bundle/'m3_labels_base', labels)
m2 = m2_output/'m3_labels_detector_v2'
for name in ['boxes_det.npy','present_mask_det.npy','detector_provenance.json']:
    shutil.copy2(m2/name, labels/name)
# Launcher preflight expects the data-level concept-space copy.
(REPO_DIR/'data').mkdir(exist_ok=True)
shutil.copy2(labels/'m3_concept_space.json', REPO_DIR/'data/m3_concept_space.json')
print('labels:', labels)
print('detector provenance:', json.loads((labels/'detector_provenance.json').read_text())['detector_checkpoint_sha256'])

In [ ]:
# Feature smoke test and manifest coverage.
import torch, numpy as np
feature_files = list(FEATURE_ROOT.rglob('*.pt')) + list(FEATURE_ROOT.rglob('*.npy'))
assert feature_files, FEATURE_ROOT
sample = feature_files[0]
x = torch.load(sample, map_location='cpu') if sample.suffix == '.pt' else np.load(sample, mmap_mode='r')
if isinstance(x, dict): x = next(v for v in x.values() if torch.is_tensor(v))
print('feature files:', len(feature_files), 'sample:', sample.name, tuple(x.shape), getattr(x,'dtype',None))
assert tuple(x.shape) in {(197,512),(196,512)}
rows = [json.loads(s) for s in (labels/'manifest.jsonl').read_text(encoding='utf-8').splitlines() if s.strip()]
ids = {str(r['image_id']) for r in rows}
feature_ids = {p.stem for p in feature_files}
print('manifest IDs:',len(ids),'missing features:',len(ids-feature_ids))
assert not (ids-feature_ids), 'feature cache does not cover manifest'

In [ ]:
# Shared launcher configuration. Conservative settings for each T4.
env = os.environ.copy()
env.update(PY='python', DEVICE='cuda:0', BATCH='8', W='2', EVAL_W='2', EP='40',
           FEAT=str(FEATURE_ROOT), LABELS=str(labels), RUNS='/kaggle/working/m3_runs',
           LOGDIR='/kaggle/working/m3_logs', DIAGDIR='/kaggle/working/m3_diagnostics',
           SYNC_REMOTE=remote+'/m3_runs', SYNC_EVERY='0')
subprocess.run(['rclone','copy',remote+'/m3_diagnostics','/kaggle/working/m3_diagnostics'],check=False)
subprocess.run(['bash',str(REPO_DIR/'phase_3/run_paper_m3_v2.sh'),'--profile','local4060','--scope','preflight'],cwd=REPO_DIR,env=env,check=True)
# Main faithful detector-box row first.
subprocess.run(['bash',str(REPO_DIR/'phase_3/run_paper_m3_v2.sh'),'--profile','local4060','--scope','main'],cwd=REPO_DIR,env=env,check=True)
subprocess.run(['rclone','copy','/kaggle/working/m3_runs/m3v2_vera_graph_lse_det',remote+'/m3_runs/m3v2_vera_graph_lse_det'],check=True)
print('main M3 checkpoint saved to Drive')

In [ ]:
# Run the remaining retained rows two at a time, one process per T4.
names = ['m3v2_no_concept_det','m3v2_concept_mlp_det','m3v2_graph_global_fusion_det',
         'm3v2_global_only_det','m3v2_graph_attention_det','m3v2_graph_mean_det',
         'm3v2_graph_max_det','m3v2_vera_graph_lse_gt']
for start in range(0, len(names), 2):
    procs=[]
    for gpu, name in enumerate(names[start:start+2]):
        e=env.copy(); e.update(DEVICE=f'cuda:{gpu}', RUN_NAME=name)
        procs.append(subprocess.Popen(['bash',str(REPO_DIR/'phase_3/run_paper_m3_v2.sh'),'--profile','local4060','--scope','all'],cwd=REPO_DIR,env=e))
    for p in procs:
        assert p.wait()==0
subprocess.run(['rclone','copy','/kaggle/working/m3_runs',remote+'/m3_runs'],check=True)
subprocess.run(['rclone','copy','/kaggle/working/m3_diagnostics',remote+'/m3_diagnostics'],check=True)
print('all retained M3 rows and diagnostics saved to Drive')